In [4]:
import tkinter as tk
from tkinter import ttk, messagebox
import pandas as pd
import joblib

metal_model = joblib.load("metal_model.joblib")
recovery_model = joblib.load("recovery_model.joblib")
cost_model = joblib.load("cost_model.joblib")

gold_price = 155.27
silver_price = 2.59
copper_price = 0.010
palladium_price = 49.58
platinum_price = 67.52

brands = [
    "Apple","Samsung","Dell","HP","Lenovo","Asus","Acer",
    "Sony","LG","Xiaomi","OnePlus","Intel","Cisco",
    "Nokia","Panasonic"
]

conditions = ["Excellent","Good","Fair","Damaged","Scrap"]

device_types = [
    "Smartphone","Laptop","Desktop","Tablet",
    "TV","Router","Server","Printer"
]

root = tk.Tk()
root.title("E-Waste Profit Predictor")
root.geometry("900x700")
root.configure(bg="#f4f6f8")

title = tk.Label(
    root,
    text="E-Waste Recycling Profit Prediction",
    font=("Arial",20,"bold"),
    bg="#f4f6f8"
)
title.pack(pady=20)

frame = tk.Frame(root, bg="#f4f6f8")
frame.pack()

def add_field(text, row, values=None):
    lbl = tk.Label(frame, text=text, font=("Arial",12), bg="#f4f6f8")
    lbl.grid(row=row, column=0, padx=15, pady=10, sticky="w")

    if values:
        box = ttk.Combobox(frame, values=values, state="readonly", width=30)
        box.grid(row=row, column=1, padx=15, pady=10)
        box.current(0)
        return box
    else:
        ent = tk.Entry(frame, width=33)
        ent.grid(row=row, column=1, padx=15, pady=10)
        return ent

brand_box = add_field("Brand Name", 0, brands)
condition_box = add_field("Device Condition", 1, conditions)
type_box = add_field("Device Type", 2, device_types)
year_entry = add_field("Year of Manufacturing", 3)

result_text = tk.Text(root, height=20, width=90, font=("Consolas",11))
result_text.pack(pady=20)

def predict():
    try:
        year = int(year_entry.get())

        
        condition_map = {
            "Excellent": 5,
            "Good": 4,
            "Fair": 3,
            "Damaged": 2,
            "Scrap": 1
        }

        condition_score = condition_map[condition_box.get()]
        device_age = 2026 - year
        

        
        data = pd.DataFrame([{
            "Brand Name": brand_box.get(),
            "Device Condition": condition_box.get(),
            "Device Type": type_box.get(),
            "Year of Manufacturing": year,
            "Device Age": device_age,
            "Condition Score": condition_score
        }])
       

        metals = metal_model.predict(data)[0]
        recovery = recovery_model.predict(data)[0]
        cost = cost_model.predict(data)[0]

        gold = metals[0]
        silver = metals[1]
        copper = metals[2]
        palladium = metals[3]
        platinum = metals[4]

        recovery_fraction = recovery / 100

        revenue = (
            gold * recovery_fraction * gold_price +
            silver * recovery_fraction * silver_price +
            copper * recovery_fraction * copper_price +
            palladium * recovery_fraction * palladium_price +
            platinum * recovery_fraction * platinum_price
        )

        profit = revenue - cost + 12

        result_text.delete("1.0", tk.END)

        result_text.insert(tk.END, "Predicted Metal Weights (grams)\n")
        result_text.insert(tk.END, "-"*45 + "\n")
        result_text.insert(tk.END, f"Gold       : {gold:.4f}\n")
        result_text.insert(tk.END, f"Silver     : {silver:.4f}\n")
        result_text.insert(tk.END, f"Copper     : {copper:.4f}\n")
        result_text.insert(tk.END, f"Palladium  : {palladium:.4f}\n")
        result_text.insert(tk.END, f"Platinum   : {platinum:.4f}\n\n")

        result_text.insert(tk.END, f"Recovery Rate     : {recovery:.2f}%\n")
        result_text.insert(tk.END, f"Recovery Cost     : ${cost:.2f}\n")
        result_text.insert(tk.END, f"Estimated Revenue : ${revenue:.2f}\n")
        result_text.insert(tk.END, f"Estimated Profit  : ${profit:.2f}\n")

   
    except Exception as e:
        messagebox.showerror("Error", str(e))
    

btn = tk.Button(
    root,
    text="Predict Profit",
    font=("Arial",13,"bold"),
    bg="#2e86de",
    fg="white",
    width=20,
    command=predict
)
btn.pack(pady=10)

root.mainloop()